# AI Copilot — Interactive Demo

This notebook walks through all three engines of the AI Copilot:
1. **Job Hunter** — live job search, CV matching, cover letter generation
2. **Code Assistant** — solve, debug, explain, review, generate, convert
3. **Knowledge Engine** — ask, deep-dive, compare, fact-check

**Prerequisites:**
- Ollama running locally with `llama3` and `nomic-embed-text` pulled
- `pip install -r requirements.txt` completed
- `configs/config.yaml` configured with your details

Run cells top to bottom. Each section is independent.

In [ ]:
import sys, os
# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

from src.utils.config_loader import load_config
from src.utils.llm_client import LLMClient

# Load config
config = load_config('../configs/config.yaml')
print(f'Candidate : {config.candidate.name}')
print(f'LLM model : {config.llm.model}')
print(f'Job queries: {len(config.job_search.queries)} configured')

# Initialise LLM client
llm = LLMClient(model=config.llm.model, temperature=config.llm.temperature)
print('\nLLM client ready.')

## Engine 1 — Code Assistant

Demonstrates: `solve`, `debug`, `explain`, `review`, `generate`, `convert`

In [ ]:
from src.agents.code_assistant import CodeAssistantAgent

code_agent = CodeAssistantAgent(
    llm_client = llm,
    output_dir = '../outputs/code_solutions'
)

# Solve a problem
solution = code_agent.solve(
    problem  = 'Write a Python function that computes battery State of Health (SOH) '
               'from a list of discharge capacities and a rated capacity, '
               'returning both current SOH and a degradation trend.',
    language = 'python',
    save     = True
)

print('\n' + '='*60)
print('GENERATED CODE:')
print('='*60)
print(solution.code)
print('\nEXPLANATION:')
print(solution.explanation)
if solution.saved_path:
    print(f'\nSaved to: {solution.saved_path}')

In [ ]:
# Debug broken code
broken_code = '''
import numpy as np

def compute_soh(capacities, rated):
    sohs = [c / rated for c in capacities]
    trend = np.polyfit(range(len(sohs)), sohs)
    return sohs[-1], trend[0]
'''

error_msg = "TypeError: polyfit() missing 1 required positional argument: 'deg'"

debug_result = code_agent.debug(
    code     = broken_code,
    error    = error_msg,
    language = 'python'
)
print('\nFixed code:')
print(debug_result.code)

In [ ]:
# Convert MATLAB to Python
matlab_code = '''
function soh = computeSOH(Q_discharge, Q_rated)
    soh = Q_discharge / Q_rated;
    if soh < 0.8
        disp('Battery reached end of life');
    end
end
'''

converted = code_agent.convert(
    code        = matlab_code,
    target_lang = 'python',
    source_lang = 'matlab'
)
print('Converted to Python:')
print(converted.code)

## Engine 2 — Knowledge Engine

Demonstrates: `ask`, `compare`, `explain_concept`, `fact_check`

In [ ]:
from src.agents.knowledge_engine import KnowledgeEngineAgent

knowledge_agent = KnowledgeEngineAgent(
    llm_client = llm,
    ntfy_topic = config.notifications.ntfy_topic
)

# Ask a technical question
response = knowledge_agent.ask(
    'What are the most common approaches to battery SOH estimation in 2026?'
)

In [ ]:
# Compare two technologies
compare_result = knowledge_agent.compare(
    option_a = 'FAISS',
    option_b = 'ChromaDB',
    context  = 'production RAG pipeline for engineering documents'
)

In [ ]:
# Explain a concept
explanation = knowledge_agent.explain_concept('Kalman filter for SOC estimation')

In [ ]:
# Fact-check a claim
fact_result = knowledge_agent.fact_check(
    'Solid-state batteries have higher energy density than lithium-ion batteries'
)

## Engine 3 — Job Hunter

Runs the full pipeline: search → score → generate cover letters.

**Note:** This makes live DuckDuckGo searches and generates PDF cover letters.
Make sure your CV PDFs are in `data/cv/` as configured in `config.yaml`.

In [ ]:
from src.agents.job_hunter import JobHunterAgent

job_agent = JobHunterAgent(
    config     = config,
    llm_client = llm
)

# Run the full pipeline
# Set notify=False to skip ntfy push notification during demo
result = job_agent.run(notify=False)

print(f'\nFound {len(result.matches)} job matches')
print(f'Cover letters generated: {len(result.cover_letters)}')
for i, match in enumerate(result.matches, 1):
    print(f'  [{i}] {match.match_score}/100 — {match.title} @ {match.company}')

In [ ]:
# Generate a single cover letter for a specific role
from src.agents.job_hunter import JobMatch, generate_cover_letter_pdf

test_match = JobMatch(
    title       = 'ML Engineer — Battery Systems',
    company     = 'ACCURE Battery Intelligence',
    snippet     = 'ML engineer to build predictive models for battery degradation',
    match_score = 88,
)

pdf_path = generate_cover_letter_pdf(
    match            = test_match,
    candidate_name   = config.candidate.name,
    candidate_email  = config.candidate.email,
    profile_summary  = config.candidate.profile_summary,
    llm_client       = llm,
    output_dir       = '../outputs/cover_letters'
)

print(f'Cover letter saved to: {pdf_path}')